# GIC 2026 Phase 3 - full qBraid reproduction

**Team:** Quantum Pattern Recognition  
**Project:** A Sector-Scalable Generative Quantum Eigensolver with Adaptive Topology and Exact Structured Circuit Synthesis  
**Track:** Mitsubishi Chemical Group / AIST - Quantum Materials Discovery Challenge

Choose a **Python 3.12** kernel and then **Run -> Run All Cells**. Use at least 4 vCPU, 8 GB RAM and 13 GiB free disk for a fresh installation. The workflow is CPU-only, credential-free, and contains no QPU-submission call. The first pinned-stack installation can take 10-30 minutes; the complete reproduction typically takes 25-60 minutes depending on qBraid load. Full mode freshly generates and validates the Table 3 candidate; no pre-existing promoted Table 3 directory is required.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

candidates = [Path.cwd().resolve(), Path.cwd().resolve() / 'Source_Code']
for parent in Path.cwd().resolve().parents:
    candidates.extend([parent, parent / 'Source_Code'])
ROOT = next((p for p in candidates if (p / 'setup.sh').is_file()), None)
assert ROOT is not None, 'Could not locate Source_Code/setup.sh. Extract the complete submission ZIP.'
sys.path.insert(0, str(ROOT))
from environment_contract import resolve_environment
VENV = resolve_environment(ROOT)
PYTHON = VENV / 'bin' / 'python'
print('Source folder:', ROOT)
print('Notebook kernel:', sys.version)


## 1. Create the pinned environment

The isolated environment reproduces the tested CUDA-QX/CUDA-Q, PyTorch, Qiskit, Aer and numerical stack without changing the qBraid system kernel.


In [ ]:
setup_env = os.environ.copy()
setup_env['QBRAID_GQE_ENV'] = str(VENV)
setup_env['PYTHON_BIN'] = sys.executable
subprocess.run(['bash', 'setup.sh'], cwd=ROOT, env=setup_env, check=True)
assert PYTHON.is_file(), f'Environment creation failed: {PYTHON}'
print('Pinned environment ready:', VENV)


## 2. Regenerate and validate the complete reported matrix

This command runs the scientific tests; frozen-input 6-48-qubit ladder; matched topology, warm-start and QSCI controls; exact structured exports; CUDA-Q checks; exact 6-qubit QPD; H2 Transformer-GQE; and finite-shot BeH2-6 experiment. Every claimed scientific quantity is checked against a declared tolerance.


In [ ]:
env = os.environ.copy()
env['QBRAID_GQE_ENV'] = str(VENV)
subprocess.run([str(PYTHON), '-I', '-B', str(ROOT / 'certify_release.py'), '--full'], cwd=ROOT, env=env, check=True)


## 3. Display the machine-readable pass/fail summary

A successful run saves both a stable latest certificate and a timestamped scientific summary. The paths printed below are the files judges should inspect.


In [ ]:
RESULTS = ROOT / 'results' / 'judge_reproduction'
certificate_path = RESULTS / 'latest_certificate.json'
latest_run_path = RESULTS / 'latest_run.json'
certificate = json.loads(certificate_path.read_text())
latest_run = json.loads(latest_run_path.read_text())
assert certificate['status'] == 'PASS', certificate
assert certificate['mode'] == 'full', certificate
assert latest_run['status'] == 'PASS', latest_run
assert latest_run['invocation_id'] == certificate['invocation_id'], (latest_run, certificate)
print(json.dumps({k: certificate[k] for k in ('status', 'mode', 'invocation_id', 'wall_seconds')}, indent=2))
print('Hash-bound generated artifacts:', certificate['generated_artifacts']['file_count'])
print('Certificate:', certificate_path)
print('Scientific summary:', latest_run['summary'])
print('ALL REQUESTED PHASE 3 RESULTS REPRODUCED WITHIN DECLARED TOLERANCES')
